<a href="https://colab.research.google.com/github/haneef5244/google-colab/blob/rag-documentation-assistant/rag_documentation_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
from google.colab import userdata
import os

# Create a .env file to store your API keys and other environment variables securely.
# This is primarily for documentation/reference, as we'll set os.environ directly.
with open('.env', 'w') as f:
    f.write('INDEX_NAME="langchain-doc-index"\n')
    f.write(f'PINECONE_API_KEY="{userdata.get('PINECONE_API_KEY')}"\n')
    f.write(f'LANGSMITH_ENDPOINT="https://api.smith.langchain.com"\n')
    f.write(f'LANGSMITH_API_KEY="{userdata.get('LANGSMITH_API_KEY')}"\n')
    f.write(f'LANGSMITH_PROJECT="rag-documentation-assistant"\n')
    f.write(f'LANGCHAIN_TRACING_V2=true\n') # Use LANGCHAIN_TRACING_V2 for newer LangChain versions
    f.write(f'GOOGLE_API_KEY="{userdata.get('GOOGLE_API_KEY')}"\n')

# Explicitly set environment variables to ensure they are picked up immediately
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")
os.environ["INDEX_NAME"] = "langchain-doc-index"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGSMITH_PROJECT"] = "rag-documentation-assistant"
os.environ["LANGCHAIN_TRACING"] = "true"
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

print(f"GOOGLE_API_KEY={os.environ.get("GOOGLE_API_KEY")}")

print(".env file created. Please replace the placeholder values with your actual API key and index name.")
print("Ensure you run this cell and then re-run the `main` function cell (`tE12tHoGtn-l`) after updating the .env file.\n\n")
!cat .env

In [ ]:
from langsmith.utils import tracing_is_enabled
print(tracing_is_enabled())          # False
os.environ["LANGCHAIN_TRACING"] = "true"
print(tracing_is_enabled())          # still False!

In [ ]:
!sudo apt-get install -y zstd

In [ ]:
# Install ollama
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Starts ollama server
!nohup ollama serve > ollama.log 2>&1 &

In [ ]:
# kills ollama serve
import subprocess

# Find the PID of the 'ollama serve' process
# We look for processes named 'ollama' and filter for 'serve'
cmd = "ps aux | grep 'ollama serve' | grep -v grep | awk '{print $2}'"
process = subprocess.run(cmd, shell=True, capture_output=True, text=True)
pids = process.stdout.strip().split('\n')

if pids and pids[0]:
    print(f"Found ollama serve process with PID(s): {', '.join(pids)}")
    for pid in pids:
        # Kill the process
        kill_cmd = f"kill -9 {pid}"
        subprocess.run(kill_cmd, shell=True)
        print(f"Killed process {pid}")
else:
    print("No 'ollama serve' process found.")


The `ollama serve` process should now be terminated. You can verify it by checking the running processes again.

In [ ]:
!ollama pull qwen3-embedding:8b
!ollama pull qwen3.5:9b
!ollama pull deepseek-r1:14b
!ollama pull qwen3.5:27b

In [ ]:
!pip install langchain langchain-ollama langchain-pinecone langchain-tavily langchain-text-splitters langchain-unstructured asyncio python-dotenv langchain-google-genai

In [ ]:
# @title
# Color codes for better logging
class Colors:
    PURPLE = "\033[95m"
    CYAN = "\033[96m"
    DARKCYAN = "\033[36m"
    BLUE = "\033[94m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    BOLD = "\033[1m"
    UNDERLINE = "\033[4m"
    END = "\033[0m"


def log_info(message: str, color: str = Colors.CYAN):
    """Log info message with color"""
    print(f"{color}ℹ️  {message}{Colors.END}")


def log_success(message: str):
    """Log success message in green"""
    print(f"{Colors.GREEN}✅ {message}{Colors.END}")


def log_error(message: str):
    """Log error message in red"""
    print(f"{Colors.RED}❌ {message}{Colors.END}")


def log_warning(message: str):
    """Log warning message in yellow"""
    print(f"{Colors.YELLOW}⚠️  {message}{Colors.END}")


def log_header(message: str):
    """Log header message with emphasis"""
    print(f"\n{Colors.BOLD}{Colors.PURPLE}{'='*60}{Colors.END}")
    print(f"{Colors.BOLD}{Colors.PURPLE}🚀 {message}{Colors.END}")
    print(f"{Colors.BOLD}{Colors.PURPLE}{'='*60}{Colors.END}\n")

# Adding Env Variables

# Crawl Pipeline

The logic below demonstrates how we can use TavilyCrawl to crawl websites

---


```max_depth``` --> how far away should Tavily crawl from given url

```max_breadth``` --> number of links Tavily are allowed to click for a given page







In [ ]:
# @title
from langchain_tavily import TavilyCrawl
from langchain_core.documents import Document

async def crawl_url(url: str):
    tavily_crawl = TavilyCrawl()

    res = await tavily_crawl.ainvoke({
        "url": url,
        "max_depth": 5,
        "extract_depth": "advanced"
    })
    all_docs = res["results"]

    results = [Document(page_content=result['raw_content'], metadata={ 'source': result['url']}) for result in all_docs]

    log_success(
        f"TavilyCrawl: Successfully crawled {len(all_docs)} URLs from {url}"
    )

    return results

**Ingestion Logic**



1.   Content from url is crawled and added as List of Document
2.   List of Document are split/chunked using `RecursiveCharacterTextSplitter`
3.   Documents are then inserted into Pinecone in batches



In [ ]:
# @title
import asyncio, os, ssl
from typing import Any, Dict, List

import certifi
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_tavily import TavilyCrawl, TavilyExtract, TavilyMap, tavily_crawl

load_dotenv()

# Configure SSL Context to use certifi certificates
ssl_context = ssl.create_default_context(cafile=certifi.where())
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

documentation_url = "https://python.langchain.com"

 # Call embedding model
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:8b",
    dimensions=1536
)

vector_store = PineconeVectorStore(
        index_name=os.environ.get("INDEX_NAME"),
        embedding=embeddings
)

async def main():
    """Main async function to orchestrate the entire process."""

    log_header("DOCUMENTATION INGESTION PIPELINE")

    log_info(
        f"   TavilyCrawl: Starting to Crawl documentation from {documentation_url}",
        Colors.PURPLE
    )

    # Load the data

    documents = await crawl_url(url=documentation_url)

    # Split the data into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=200)
    splitted_docs = text_splitter.split_documents(documents)
    log_success(
        f"Text Splitter: Created {len(splitted_docs)} chunks from {len(documents)} documents"
    )

    await aindex_documents(documents=splitted_docs, batch_size=200)

async def aindex_documents(documents: List[Document], batch_size: int = 50):
    """Process documents in batches asynchronously"""
    log_header("VECTOR STORAGE PHASE")
    log_info(
        f"  VectorStore indexing: Preparing to add {len(documents)} documents to vector store",
        Colors.DARKCYAN
    )

    batches = [
        documents[i : i + batch_size] for i in range(0, len(documents), batch_size)
    ]

    log_info(
        f"  VectorStore indexing: split into {len(batches)} batches of {batch_size} documents each"
    )

    async def aadd_batch(batches: List[Document], batch_num: int):
        try:
            await vector_store.aadd_documents(batches)
            log_success(
                f"VectorStore indexing: Successfully added batch {batch_num}/{len(batches)} ({len(batches)} documents)"
            )
        except Exception as e:
            log_error(f"VectorStore indexing: Failed to add batch {batch_num} - {e}")
            return False
        return True

    tasks = [aadd_batch(batch, i + 1) for i, batch in enumerate(batches)]
    results = await asyncio.gather(*tasks, return_exceptions=True)

    successful_count = sum(1 for i in results if i == True)

    log_info(
        f"Inserted {successful_count}/{len(results)}"
    )

await main()

# Retrieval pipeline

In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import ToolMessage
from langchain.tools import tool

from langchain_pinecone import PineconeVectorStore
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.messages import HumanMessage

from typing import Any, Dict

llm_model = ChatOllama(model="deepseek-r1:14b")
# llm_model = ChatGoogleGenerativeAI(model="gemini-3.8-flash")
embeddings = OllamaEmbeddings(model="qwen3-embedding:8b", dimensions=1536)
# embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", output_dimensionality=1536)




# content_and_artifact returns 2 values
@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve relevant documentation to help answer user queries about LangChain."""

    vector_store = PineconeVectorStore(index_name=os.environ.get("INDEX_NAME"), embedding=embeddings)
    results = vector_store.as_retriever(search_kwargs={'k':5}).invoke(query)

    # Serialize documents for the model
    serialized = "\n\n".join(
        (f"Source: {doc.metadata.get("source", "Unknown")}\n\nContent: {doc.page_content}")
        for doc in results
    )

    # Return both serialized content and raw documents
    return serialized, results

def run_llm(query: str) -> Dict[str, Any]:
    """
    Run the RAG pipeline to answer a query using retrieved documentation.

    Args:
        query: The user's question

    Returns:
        Dictionary containing:
            - answer: The generated answer
            - context: The list of retrieved documents
    """

    print(f"Query={query}")
    system_prompt = """
        You are a helpful AI assistant that answers questions about LangChain documentation.
        You have access to a tool that retrieves relevant documentation.
        You MUST use the tool to find relevant information before answering questions.
        Always cite the sources you use in our answers.
        If you cannot find the answer in the retrieved documentation, say so.
    """
    agent = create_agent(model=llm_model, tools=[retrieve_context], system_prompt=system_prompt)
    # result = agent.invoke({"messages": [{"role": "user", "content": "Summarize AI trends"}]})
    response = agent.invoke({"messages": [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]})
    print()
    print(response)
    answer = response["messages"][-1].content
    return answer

def main():
    run_llm("What is the latest LangChain function?")

if __name__ == "__main__":
    main()

